# CCI Session 2 — Lab 6: Clinical Data Extraction with GPT-4.1-mini & Email Report

**Objective:** Use OpenAI's GPT-4.1-mini to extract structured clinical data from a patient discharge note, then email the JSON result via Gmail SMTP.

**Notebook follows the 9-section KHCC template.**

---
## Section 1: Package Installations

In [ ]:
!pip install openai -q

---
## Section 2: Imports

In [ ]:
import json
import csv
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime
from openai import OpenAI
from google.colab import userdata

---
## Section 3: Configuration

In [ ]:
# OpenAI — key stored in Colab Secrets (key icon in left sidebar)
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"

# Gmail SMTP — use an App Password, NOT your regular password
# Generate one at: https://myaccount.google.com/apppasswords
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = userdata.get("GMAIL_ADDRESS")       # e.g. yourname@gmail.com
SENDER_APP_PASSWORD = userdata.get("GMAIL_APP_PASSWORD")  # 16-char app password
RECIPIENT_EMAIL = "iyad.y.sultan@gmail.com"  # <-- change to desired recipient

REPORT_DATE = datetime.now().strftime("%Y-%m-%d %H:%M")
OUTPUT_CSV = "extracted_patient_data.csv"

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Config loaded — model: {MODEL}, date: {REPORT_DATE}")

---
## Section 4: Prompts

In [ ]:
SYSTEM_PROMPT = """You are a clinical data extraction assistant at King Hussein Cancer Center.
Extract structured data from the patient note provided by the user.
Return ONLY valid JSON with no markdown fences and no extra text."""

EXTRACTION_PROMPT = """Extract the following fields from the patient note below.
Return a single JSON object with exactly these keys:

{
  "patient_name": "string",
  "mrn": "string",
  "age": integer,
  "gender": "string",
  "primary_diagnosis": "string",
  "stage": "string or null",
  "hemoglobin": float or null,
  "wbc": float or null,
  "platelets": integer or null,
  "creatinine": float or null,
  "medications": ["list of strings"],
  "allergies": ["list of strings"],
  "procedures": ["list of strings"],
  "discharge_disposition": "string",
  "follow_up": "string"
}

If a field is not mentioned in the note, use null.

--- PATIENT NOTE ---
{note}
--- END NOTE ---"""

---
## Section 5: Functions

In [ ]:
def extract_patient_data(note_text: str) -> dict:
    """Send a patient note to GPT-4.1-mini and return structured JSON."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": EXTRACTION_PROMPT.format(note=note_text)},
        ],
        temperature=0.0,
    )
    raw = response.choices[0].message.content.strip()
    # Strip markdown fences if the model adds them anyway
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)


def build_email_body(data: dict) -> str:
    """Format extracted data into a readable email body."""
    lines = [
        f"Clinical Data Extraction Report — {REPORT_DATE}",
        "=" * 50,
        f"Patient : {data.get('patient_name', 'N/A')}",
        f"MRN     : {data.get('mrn', 'N/A')}",
        f"Age/Sex : {data.get('age', 'N/A')} / {data.get('gender', 'N/A')}",
        f"Diagnosis: {data.get('primary_diagnosis', 'N/A')} (Stage {data.get('stage', 'N/A')})",
        "",
        "Lab Values",
        "-" * 30,
        f"  Hemoglobin : {data.get('hemoglobin', 'N/A')} g/dL",
        f"  WBC        : {data.get('wbc', 'N/A')} K/uL",
        f"  Platelets  : {data.get('platelets', 'N/A')} K/uL",
        f"  Creatinine : {data.get('creatinine', 'N/A')} mg/dL",
        "",
        f"Medications: {', '.join(data.get('medications', [])) or 'None'}",
        f"Allergies  : {', '.join(data.get('allergies', [])) or 'None'}",
        f"Procedures : {', '.join(data.get('procedures', [])) or 'None'}",
        "",
        f"Discharge: {data.get('discharge_disposition', 'N/A')}",
        f"Follow-up: {data.get('follow_up', 'N/A')}",
        "",
        "--- Full JSON attached below ---",
        json.dumps(data, indent=2, ensure_ascii=False),
    ]
    return "\n".join(lines)


def send_email(recipient: str, subject: str, body: str):
    """Send a plain-text email via Gmail SMTP with TLS."""
    msg = MIMEMultipart()
    msg["From"] = SENDER_EMAIL
    msg["To"] = recipient
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()
        server.login(SENDER_EMAIL, SENDER_APP_PASSWORD)
        server.send_message(msg)

    print(f"Email sent to {recipient}")

---
## Section 6: Main Code — Sample Patient Note + Extraction

In [ ]:
SAMPLE_NOTE = """
DISCHARGE SUMMARY
King Hussein Cancer Center — Department of Medical Oncology

Patient Name: Ahmad Khalil Al-Rashid
MRN: P-20458
Age: 62 years   Sex: Male
Date of Admission: 2026-03-15
Date of Discharge: 2026-03-22

PRIMARY DIAGNOSIS: Non-Small Cell Lung Cancer (Adenocarcinoma), Stage IIIB

HISTORY OF PRESENT ILLNESS:
Mr. Al-Rashid is a 62-year-old male with a history of NSCLC diagnosed in January 2026
after presenting with persistent cough and hemoptysis. CT chest showed a 4.2 cm right
upper lobe mass with mediastinal lymphadenopathy. Biopsy confirmed adenocarcinoma,
EGFR wild-type, ALK negative, PD-L1 TPS 65%.

HOSPITAL COURSE:
Patient admitted for Cycle 1 of Pembrolizumab 200 mg IV + Carboplatin AUC 5 +
Pemetrexed 500 mg/m2. Tolerated infusion well. Developed Grade 1 nausea managed
with Ondansetron. No febrile neutropenia.

LABORATORY DATA (Discharge):
  Hemoglobin: 11.2 g/dL
  WBC: 5.8 K/uL
  Platelets: 198 K/uL
  Creatinine: 0.9 mg/dL
  ALT: 22 U/L
  AST: 18 U/L

MEDICATIONS AT DISCHARGE:
1. Ondansetron 8 mg PO PRN nausea
2. Dexamethasone 4 mg PO daily x 3 days
3. Pantoprazole 40 mg PO daily
4. Filgrastim 300 mcg SC daily x 5 days (if ANC < 1.5)

ALLERGIES: Sulfonamides (rash), Ibuprofen (GI bleeding)

PROCEDURES DURING ADMISSION:
- Port-a-Cath insertion (right subclavian, IR-guided)
- CT-guided core biopsy of right upper lobe mass (prior admission)

DISCHARGE DISPOSITION: Home in stable condition

FOLLOW-UP:
Medical Oncology clinic in 21 days for Cycle 2. CBC and CMP 7 days prior.
CT chest/abdomen/pelvis after Cycle 3 for response assessment.

Attending Physician: Dr. Nadia Haddad, MD
Department of Medical Oncology, KHCC
"""

print("Sending note to GPT-4.1-mini for extraction...")
extracted = extract_patient_data(SAMPLE_NOTE)
print("\nExtracted JSON:")
print(json.dumps(extracted, indent=2, ensure_ascii=False))

---
## Section 7: Build CSV

In [ ]:
flat = {
    k: (", ".join(v) if isinstance(v, list) else v)
    for k, v in extracted.items()
}

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=flat.keys())
    writer.writeheader()
    writer.writerow(flat)

print(f"Saved to {OUTPUT_CSV}")

---
## Section 8: Email Results

In [ ]:
subject = f"[KHCC-CCI] Patient Data Extraction — {extracted.get('patient_name', 'Unknown')} ({REPORT_DATE})"
body = build_email_body(extracted)

print("Preview of email body:")
print("=" * 50)
print(body)
print("=" * 50)

# Uncomment the line below to actually send the email.
# Make sure GMAIL_ADDRESS and GMAIL_APP_PASSWORD are set in Colab Secrets.
# send_email(RECIPIENT_EMAIL, subject, body)

print(f"\nTo send: uncomment send_email() above. Recipient: {RECIPIENT_EMAIL}")

---
## Section 9: Markdown Summary

| Item | Value |
|------|-------|
| **Purpose** | Extract structured clinical data from a discharge note using GPT-4.1-mini, then email the result |
| **Model** | gpt-4.1-mini (OpenAI) |
| **Input** | 1 synthetic KHCC discharge summary |
| **Output** | `extracted_patient_data.csv` + email report |
| **Fields extracted** | 15 (name, MRN, age, gender, diagnosis, stage, 4 labs, meds, allergies, procedures, disposition, follow-up) |
| **Email** | Gmail SMTP with TLS, App Password auth |
| **Author** | [Your Name] |